In [ ]:
import joblib
from dask.distributed import Client as DaskClient
from sklearn.ensemble import RandomForestClassifier

from utils import (
    BBOX,
    BBOX_GDF,
    DATETIME,
    MODE,
    classes,
    load_data,
    load_embeddings,
    training_data,
)

In [ ]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [ ]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

In [ ]:
BBOX_GDF.explore()

In [ ]:
if MODE == "traditional":
    merged = load_data(BBOX, DATETIME)
elif MODE == "embeddings":
    merged = load_embeddings(BBOX, DATETIME)

In [ ]:
merged

## Train and predict

When you change your training data, you can re-train and predict here.

In [ ]:
print(f"Number of training samples: {len(training_data)}")

# print(training_data.crs)
# print(merged.odc.crs)

# Reproject 4326 training data to 3832 like the main dataset
training_data = training_data.to_crs(merged.odc.crs)

print("Distribution of lulc_code in the training data:")
print(training_data["lulc_code"].value_counts().sort_index())

# Colour points by class, legend shows "code: label"
labels = {code: f"{code}: {name}" for code, name, _ in classes}
training_data.assign(lulc_class=training_data["lulc_code"].map(labels)).explore(
    column="lulc_class",
    categorical=True,
    categories=list(labels.values()),
    cmap=[color for *_, color in classes],
    legend=True,
)

In [ ]:
# Sample merged at each training point (nearest pixel)
points = training_data.get_coordinates().to_xarray()
samples = merged.sel(
    x=points.x,
    y=points.y,
    method="nearest",
    # Finally load the merged data. Until now it was lazy
).compute()
training_array = training_data.join(samples.to_dataframe())

# Keep only the lulc code and model features
features = list(merged.data_vars)
training_array = training_array[["lulc_code", *features]]

# Drop points that fall on NaN (masked) pixels
n_points = len(training_array)
training_array = training_array.dropna(how="all", subset=features)
print(
    f"Training array: {training_array.shape}, dropped {n_points - len(training_array)} NaN rows"
)

In [ ]:
training_array.to_csv(f"training_array_{MODE}.csv", index=False)
training_array

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

# DataFrame keeps feature names on the model
model = classifier.fit(training_array[features], training_array["lulc_code"])

In [ ]:
print(classifier.oob_score_)

In [ ]:
joblib.dump(model, f"test_model_{MODE}.dump")

In [ ]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(features, classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    print(f"{i[0]:<11}| {i[1]:.2f}")